# Bodhan Evaluation Harness Walkthrough

This notebook demonstrates how to use `bodhan_eval` to audit translation and speech quality across Indian languages.

### Pipeline Overview
1. **Load Dataset:** Load items with source texts, translations, and optional audio.
2. **Deterministic Checks:** Run free checks for numerals, formatting, scripts, and entities.
3. **Model & Speech Checks:** Back-translation and audio signal / back-transcription checks.
4. **Reporting:** Inspect executive summary and item-level flags with character spans.

## 1. Setup & Imports

In [ ]:
import json
from pathlib import Path
from bodhan_eval.config import Config
from bodhan_eval.orchestrator import Orchestrator
from bodhan_eval.normalize import extract_numbers, normalize_indic_numerals

## 2. Inspecting Script & Numeral Normalization

The harness normalizes Indic numerals across all scripts (Devanagari, Bengali, Tamil, etc.) to ASCII digits while preserving character spans.

In [ ]:
# Devanagari Hindi numeral normalization
hindi_sample = "पानी १०० °C पर उबलता है।"
print("Normalized:", normalize_indic_numerals(hindi_sample))

# Extracted numbers with spans
matches = extract_numbers(hindi_sample)
for m in matches:
    print(f"Value: {m.numeric_value}, Span: {m.span}, Raw: '{m.raw}'")

## 3. Running the Quality Evaluation Harness

Execute the evaluation harness on the provided demo dataset.

In [ ]:
config = Config(
    dataset="data/demo.jsonl",
    langs="hi,ta,bn",
    out="./report",
    calibration="data/calibration_labels.jsonl",
    max_cost_inr=50.0,
)

orchestrator = Orchestrator(config)
await orchestrator.run()

## 4. Reading the Generated Reports

In [ ]:
summary_md_path = Path("report/summary.md")
if summary_md_path.exists():
    print(summary_md_path.read_text(encoding="utf-8")[:1500])